# Label Studio → YOLO Dataset Pipeline

Converts a Label Studio video object-tracking export (`seed-annotation.json`) into a
YOLO-format image dataset ready for training with **Ultralytics YOLOv8**.

### Steps
1. Parse the JSON and extract every **keyframe** bounding box (plus linear interpolation between keyframes).
2. Resolve each annotation to the matching `.mp4` chunk in `labelstudiochunks/`.
3. Extract the annotated frames from the videos as `.jpg` images.
4. Write YOLO-format `.txt` label files alongside the images.
5. Generate a `data.yaml` so YOLOv8 can train directly from the output folder.

## 0 — Configuration

In [4]:
from pathlib import Path

# ── paths ────────────────────────────────────────────────────────────────────
REPO_ROOT        = Path("../")                                        # adjust if needed
ANNOTATIONS_JSON = REPO_ROOT / "data/labels/seed-annotation.json"
CHUNKS_DIR       = REPO_ROOT / "data/processed/labelstudiochunks"
OUTPUT_DIR       = REPO_ROOT / "data/yolo_dataset"

data_yaml_path = OUTPUT_DIR / "data.yaml"


# ── interpolation ────────────────────────────────────────────────────────────
# True  → linearly interpolate bboxes between every pair of keyframes
# False → only extract the explicitly annotated keyframes
INTERPOLATE_KEYFRAMES = False

# ── dataset split ────────────────────────────────────────────────────────────
# fraction of tasks used for validation (rest → train)
VAL_SPLIT = 0.15

# ── YOLO class list  (order determines class id) ─────────────────────────────
# These must match the label names used in Label Studio exactly.
CLASSES = ["Boost", "Charge", "Defense", "Glide", "HP",
           "Offense", "Top Speed", "Turn", "Weight"]

print("Config OK")
print(f"  annotations : {ANNOTATIONS_JSON}")
print(f"  chunks dir  : {CHUNKS_DIR}")
print(f"  output dir  : {OUTPUT_DIR}")
print(f"  classes     : {CLASSES}")

Config OK
  annotations : ../data/labels/seed-annotation.json
  chunks dir  : ../data/processed/labelstudiochunks
  output dir  : ../data/yolo_dataset
  classes     : ['Boost', 'Charge', 'Defense', 'Glide', 'HP', 'Offense', 'Top Speed', 'Turn', 'Weight']


## 1 — Imports & helpers

In [4]:
import json
import re
import random
import shutil
from collections import defaultdict
from pathlib import Path

import cv2
from tqdm.auto import tqdm


# ── helpers ───────────────────────────────────────────────────────────────────

def get_class_id(label: str) -> int:
    """Return the integer class id for a label string."""
    if label not in CLASSES:
        raise ValueError(f"Unknown label '{label}'. Add it to CLASSES.")
    return CLASSES.index(label)


def interpolate_sequence(sequence: list, interpolate: bool = True) -> dict:
    """
    Build {frame_number: bbox_dict} from a Label Studio videorectangle sequence.

    If interpolate=True (controlled by INTERPOLATE_KEYFRAMES at the top of the
    notebook), bboxes are linearly interpolated between every pair of keyframes
    so every frame in the annotated range gets a label.

    If interpolate=False, only the explicitly annotated keyframes are returned.
    """
    keyframes = sorted(
        [kf for kf in sequence if kf.get("enabled", True)],
        key=lambda kf: kf["frame"]
    )

    frame_map: dict = {}

    for i, kf in enumerate(keyframes):
        frame_map[kf["frame"]] = kf

        if interpolate and i + 1 < len(keyframes):
            nf     = keyframes[i + 1]
            f0, f1 = kf["frame"], nf["frame"]
            steps  = f1 - f0
            for step in range(1, steps):
                t = step / steps
                frame_map[f0 + step] = {
                    "frame":   f0 + step,
                    "x":       kf["x"]      + t * (nf["x"]      - kf["x"]),
                    "y":       kf["y"]      + t * (nf["y"]       - kf["y"]),
                    "width":   kf["width"]  + t * (nf["width"]  - kf["width"]),
                    "height":  kf["height"] + t * (nf["height"] - kf["height"]),
                    "enabled": True,
                }

    return frame_map


def video_filename_from_url(url: str) -> str | None:
    """
    Extract the bare filename (e.g. 'AY_G1_chunk_0003.mp4') from a
    Label Studio local-file URL such as:
      /data/local-files/?d=Users/.../labelstudiochunks/AY_G1_chunk_0003.mp4
    """
    match = re.search(r"([^/]+\.mp4)", url, re.IGNORECASE)
    return match.group(1) if match else None


print("Helpers ready.")

Helpers ready.


## 2 — Parse annotations

In [5]:
with open(ANNOTATIONS_JSON) as f:
    tasks = json.load(f)

print(f"Loaded {len(tasks)} tasks from {ANNOTATIONS_JSON.name}")

# ── build per-task annotation table ──────────────────────────────────────────
# parsed_tasks is a list of dicts:
#   { 'video_filename': str,
#     'frame_annotations': {frame_num: [yolo_line, ...]} }

parsed_tasks = []
skipped      = []

for task in tasks:
    video_url      = task.get("data", {}).get("video", "")
    video_filename = video_filename_from_url(video_url)

    if not video_filename:
        print(f"[SKIP] Cannot parse video URL: {video_url!r}")
        skipped.append(video_url)
        continue

    # frame_annotations[frame_num] = ["class cx cy w h", ...]
    frame_annotations: dict[int, list[str]] = defaultdict(list)

    for annotation in task.get("annotations", []):
        for result in annotation.get("result", []):
            if result.get("type") != "videorectangle":
                continue

            value    = result["value"]
            labels   = value.get("labels", [])
            sequence = value.get("sequence", [])

            if not labels:
                print(f"[WARN] Task {task.get('id')}: result with no labels – skipping result.")
                continue

            class_id  = get_class_id(labels[0])
            frame_map = interpolate_sequence(sequence, interpolate=INTERPOLATE_KEYFRAMES)

            for frame_num, kf in frame_map.items():
                if not kf.get("enabled", True):
                    continue

                # Label Studio x/y/width/height are in % of frame dimensions
                cx = (kf["x"] + kf["width"]  / 2) / 100
                cy = (kf["y"] + kf["height"] / 2) / 100
                w  =  kf["width"]  / 100
                h  =  kf["height"] / 100

                # clamp to [0, 1] to guard against out-of-bounds annotations
                cx = max(0.0, min(1.0, cx))
                cy = max(0.0, min(1.0, cy))
                w  = max(0.0, min(1.0, w))
                h  = max(0.0, min(1.0, h))

                frame_annotations[frame_num].append(
                    f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"
                )

    if not frame_annotations:
        print(f"[INFO] No annotations in task {task.get('id')} ({video_filename}) – skipping.")
        continue

    parsed_tasks.append({
        "video_filename":    video_filename,
        "frame_annotations": dict(frame_annotations),
    })

print(f"\nParsed {len(parsed_tasks)} annotated tasks  ({len(skipped)} skipped).")

# Quick label-count summary
label_counts = defaultdict(int)
for pt in parsed_tasks:
    for lines in pt["frame_annotations"].values():
        for line in lines:
            cid = int(line.split()[0])
            label_counts[CLASSES[cid]] += 1

print("\nAnnotation counts per class:")
for cls, cnt in sorted(label_counts.items()):
    print(f"  {cls:<12} {cnt}")

Loaded 59 tasks from seed-annotation.json
[INFO] No annotations in task 13283 (AY_G1_chunk_0001.mp4) – skipping.
[INFO] No annotations in task 13467 (ParTwo_G1_chunk_0000.mp4) – skipping.

Parsed 57 annotated tasks  (0 skipped).

Annotation counts per class:
  Boost        371
  Charge       238
  Defense      270
  Glide        287
  HP           184
  Offense      487
  Top Speed    406
  Turn         446
  Weight       281


## 3 — Train / val split

In [6]:
random.seed(42)
shuffled = parsed_tasks.copy()
random.shuffle(shuffled)

n_val   = max(1, int(len(shuffled) * VAL_SPLIT))
val_set = shuffled[:n_val]
trn_set = shuffled[n_val:]

print(f"Train tasks : {len(trn_set)}")
print(f"Val   tasks : {len(val_set)}")

Train tasks : 49
Val   tasks : 8


## 4 — Extract frames & write YOLO labels

In [7]:
def extract_and_write(
    task_list: list,
    split: str,           # "train" or "val"
    output_root: Path,
    chunks_dir: Path,
) -> dict:
    """Extract annotated frames from videos and write YOLO labels.
    Returns a summary dict.
    """
    img_dir = output_root / "images" / split
    lbl_dir = output_root / "labels" / split
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    total_frames_saved = 0
    errors             = []

    for task in tqdm(task_list, desc=f"{split}", unit="task"):
        video_filename    = task["video_filename"]
        frame_annotations = task["frame_annotations"]
        video_path        = chunks_dir / video_filename

        if not video_path.exists():
            print(f"[ERROR] Video not found: {video_path}")
            errors.append(str(video_path))
            continue

        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            print(f"[ERROR] Cannot open: {video_path}")
            errors.append(str(video_path))
            continue

        total_video_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        stem               = video_path.stem   # e.g. AY_G1_chunk_0003

        for frame_num in sorted(frame_annotations.keys()):
            # Label Studio frames are 1-indexed; OpenCV is 0-indexed
            zero_idx = frame_num - 1
            if zero_idx < 0 or zero_idx >= total_video_frames:
                continue

            cap.set(cv2.CAP_PROP_POS_FRAMES, zero_idx)
            ret, frame = cap.read()
            if not ret:
                print(f"[WARN] Could not read frame {frame_num} from {video_filename}")
                continue

            base_name  = f"{stem}_frame{frame_num:06d}"
            img_path   = img_dir / f"{base_name}.jpg"
            lbl_path   = lbl_dir / f"{base_name}.txt"

            cv2.imwrite(str(img_path), frame, [cv2.IMWRITE_JPEG_QUALITY, 95])

            with open(lbl_path, "w") as lf:
                lf.write("\n".join(frame_annotations[frame_num]))

            total_frames_saved += 1

        cap.release()

    return {"split": split, "frames_saved": total_frames_saved, "errors": errors}


# ── run extraction ────────────────────────────────────────────────────────────
if OUTPUT_DIR.exists():
    print(f"Clearing existing output dir: {OUTPUT_DIR}")
    shutil.rmtree(OUTPUT_DIR)

results = []
for task_list, split in [(trn_set, "train"), (val_set, "val")]:
    r = extract_and_write(task_list, split, OUTPUT_DIR, CHUNKS_DIR)
    results.append(r)
    print(f"{split}: {r['frames_saved']} frames saved, {len(r['errors'])} errors")

total = sum(r["frames_saved"] for r in results)
print(f"\nTotal frames written: {total}")

train:   0%|          | 0/49 [00:00<?, ?task/s]

train: 1994 frames saved, 0 errors


val:   0%|          | 0/8 [00:00<?, ?task/s]

val: 347 frames saved, 0 errors

Total frames written: 2341


## 5 — Write supporting files

In [ ]:
import yaml  # pip install pyyaml

# ── classes.txt ───────────────────────────────────────────────────────────────
classes_txt = OUTPUT_DIR / "classes.txt"
classes_txt.write_text("\n".join(CLASSES) + "\n")
print(f"Written: {classes_txt}")

# ── data.yaml (YOLOv8 / Ultralytics format) ──────────────────────────────────
data_yaml_content = {
    "path":  str(OUTPUT_DIR.resolve()),
    "train": "images/train",
    "val":   "images/val",
    "nc":    len(CLASSES),
    "names": CLASSES,
}

with open(data_yaml_path, "w") as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False, sort_keys=False)

print(f"Written: {data_yaml_path}")
print()
print(data_yaml_path.read_text())

## 6 — Dataset summary

In [5]:
from collections import Counter

def count_labels(labels_dir: Path) -> Counter:
    c = Counter()
    for txt in labels_dir.glob("*.txt"):
        for line in txt.read_text().splitlines():
            parts = line.strip().split()
            if parts:
                c[CLASSES[int(parts[0])]] += 1
    return c

print("=" * 50)
print(f"{'Split':<8} {'Images':>8} {'Labels':>8}")
print("=" * 50)

for split in ["train", "val"]:
    img_count = len(list((OUTPUT_DIR / "images" / split).glob("*.jpg")))
    lbl_count = sum(count_labels(OUTPUT_DIR / "labels" / split).values())
    print(f"{split:<8} {img_count:>8} {lbl_count:>8}")

print("=" * 50)
print()
print("Class distribution (all splits combined):")
combined = Counter()
for split in ["train", "val"]:
    combined.update(count_labels(OUTPUT_DIR / "labels" / split))
for cls in CLASSES:
    print(f"  {cls:<12} {combined.get(cls, 0):>6}")

Split      Images   Labels
train        1994     2556
val           347      414

Class distribution (all splits combined):
  Boost           371
  Charge          238
  Defense         270
  Glide           287
  HP              184
  Offense         487
  Top Speed       406
  Turn            446
  Weight          281


## 7 — (Optional) Quick-train with YOLOv8

Run the cell below to kick off a short training run to verify everything is wired up correctly.
Install Ultralytics first if needed: `pip install ultralytics`

In [6]:
# Uncomment to run a smoke-test training (3 epochs, small model)
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
model.train(
    data=str(data_yaml_path),
    epochs=3,
    imgsz=640,
    batch=16,
    project=str(REPO_ROOT / "runs"),
    name="kartector_smoke",
)

Ultralytics 8.4.40 🚀 Python-3.12.8 torch-2.11.0 CPU (Apple M4 Pro)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../data/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=kartector_smoke, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspe

KeyboardInterrupt: 